# FMA analysis

This notebook has:

- lyrics retrieval from FMA
- Lyrics comparisson to TwitterPD
- On the selected songs, lyrics are compared to raw audio

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata
from urllib.parse import quote
import requests
from time import sleep
import sqlite3

# Preprocessing

This method proved to not be reliable. The run time is superior to 10 hours!

In [3]:
tracks_path = Path('/home/filipepimentel/Desktop/Thesis code/fma_metadata/fma_metadata/tracks.csv')

tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1], low_memory=False)

small_df = tracks[tracks[('set', 'subset')] == 'small'].copy()

def normalize_text(s: str):
    if pd.isna(s):
        return ''
    s = str(s).lower().strip()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))

    s = re.sub(r"\bfeat\.?\b.*", "", s)
    s = re.sub(r"\bft\.?\b.*", "", s)
    s = re.sub(r"\(.*?remix.*?\)", "", s)
    s = re.sub(r"\(.*?live.*?\)", "", s)
    s = re.sub(r"\[.*?\]", "", s)
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

small_df[("artist_clean", "")] = small_df[("artist", "name")].apply(normalize_text)
small_df[("title_clean", "")] = small_df[("track", "title")].apply(normalize_text)
small_df[("query_key", "")] = small_df[("artist_clean", "")] + " | " + small_df[("title_clean", "")]

print(small_df[[("artist", "name"), ("track", "title"), ("query_key", "")]].head())
print("Total FMA small tracks:", len(small_df))

                                   artist               track  \
                                     name               title   
track_id                                                        
2                                    AWOL                Food   
5                                    AWOL          This World   
10                              Kurt Vile             Freeway   
140       Alec K. Redfearn & the Eyesores  Queen Of The Wires   
141       Alec K. Redfearn & the Eyesores                Ohio   

                                                  query_key  
                                                             
track_id                                                     
2                                               awol | food  
5                                         awol | this world  
10                                      kurt vile | freeway  
140       alec k redfearn the eyesores | queen of the wires  
141                     alec k redfearn the e

Prepare APIs for the lyrics

In [10]:
def get_lyrics_ovh(artist: str, title: str, timeout=5):
    url = f'https://api.lyrics.ovh/v1/{quote(artist)}/{quote(title)}'
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            lyrics = data.get('lyrics', '').strip()
            if lyrics:
                return {
                    'found': True,
                    'source': 'lyrics.ovh',
                    'lyrics': lyrics
                }
        return {'found': False, 'source': 'lyrics.ovh', 'lyrics': None}
    except Exception:
        return {'found': False, 'source': 'lyrics.ovh', 'lyrics': None}

def search_lrclib(query: str, timeout=5):
    url = 'https://lrclib.net/api/search'
    try:
        r = requests.get(url, params={'q': query}, timeout=timeout)
        if r.status_code == 200:
            results = r.json()
            if results and isinstance(results, list):
                best = results[0]
                lyrics = best.get('plainLyrics') or best.get('plainlyrics') or best.get('syncedLyrics')
                if lyrics:
                    return {
                        'found': True,
                        'source': 'lrclib',
                        'lyrics': lyrics,
                        'matched_track': best.get('trackName'),
                        'matched_artist': best.get('artistName'),
                    }
        return {'found': False, 'source': 'lrclib', 'lyrics': None}
    except Exception:
        return {'found': False, 'source': 'lrclib', 'lyrics': None}


def find_lyrics(artist: str, title: str):
    artist_c = normalize_text(artist)
    title_c = normalize_text(title)

    result = get_lyrics_ovh(artist_c, title_c)
    if result['found']:
        return result

    query = f'{artist_c} {title_c}'
    result = search_lrclib(query)
    if result['found']:
        return result

    return {'found': False, 'source': None, 'lyrics': None}

Get the lyrics

In [11]:
checkpoint_path = Path('fma_small_lyrics_status_checkpoint.csv')
max_unique = 50  # Set to None to process all unique queries
save_every = 200

work_df = small_df[[('artist', 'name'), ('track', 'title'), ('query_key', '')]].copy()
work_df.columns = ['artist', 'title', 'query_key']
work_df = work_df.reset_index().rename(columns={'index': 'track_id'})

unique_queries = work_df[['artist', 'title', 'query_key']].drop_duplicates(subset=['query_key']).copy()
if max_unique is not None:
    unique_queries = unique_queries.head(max_unique)

completed = set()
results_unique = []

if checkpoint_path.exists():
    checkpoint_df = pd.read_csv(checkpoint_path)
    if not checkpoint_df.empty and 'query_key' in checkpoint_df.columns:
        completed = set(checkpoint_df['query_key'].astype(str))
        results_unique.extend(checkpoint_df.to_dict('records'))
        print(f'Loaded checkpoint with {len(completed)} completed queries')

pending = unique_queries[~unique_queries['query_key'].isin(completed)].copy()
print(f'Unique queries total: {len(unique_queries)} | Pending: {len(pending)}')

for i, (_, row) in enumerate(pending.iterrows(), start=1):
    artist = row['artist']
    title = row['title']
    query_key = row['query_key']

    res = find_lyrics(artist, title)

    results_unique.append({
        'query_key': query_key,
        'lyrics_found': res['found'],
        'lyrics_source': res['source'],
        'lyrics': res['lyrics']
    })

    if i % save_every == 0:
        pd.DataFrame(results_unique).drop_duplicates(subset=['query_key'], keep='last').to_csv(checkpoint_path, index=False)
        print(f'Progress: {i}/{len(pending)} pending queries processed')

results_unique_df = pd.DataFrame(results_unique).drop_duplicates(subset=['query_key'], keep='last')
results_df = work_df.merge(results_unique_df, on='query_key', how='left')

results_df.to_csv('fma_small_lyrics_status.csv', index=False)
print(results_df[['track_id', 'artist', 'title', 'lyrics_found', 'lyrics_source']].head())
print('Saved:', 'fma_small_lyrics_status.csv')
print('Rows:', len(results_df), '| Unique queries checked:', len(results_unique_df))

Unique queries total: 50 | Pending: 50
   track_id                           artist               title lyrics_found  \
0         2                             AWOL                Food        False   
1         5                             AWOL          This World        False   
2        10                        Kurt Vile             Freeway         True   
3       140  Alec K. Redfearn & the Eyesores  Queen Of The Wires        False   
4       141  Alec K. Redfearn & the Eyesores                Ohio        False   

  lyrics_source  
0           NaN  
1           NaN  
2    lyrics.ovh  
3           NaN  
4           NaN  
Saved: fma_small_lyrics_status.csv
Rows: 8000 | Unique queries checked: 50


# Preprocessing 2

In this section, I'm using a dump from the same API to achieve a superior runtime

In [15]:
import sqlite3
import gzip
import shutil

compressed_db = Path('/home/filipepimentel/Desktop/lrclib-db-dump-20260305T042906Z.sqlite3.gz')
db_path = Path('/home/filipepimentel/Desktop/Thesis code/lrclib-db-dump-20260305T042906Z.sqlite3')

if not compressed_db.exists():
    raise FileNotFoundError(f'Compressed DB not found: {compressed_db}')
if compressed_db.is_dir():
    raise IsADirectoryError(f'Expected a .gz file but got a directory: {compressed_db}')

if not db_path.exists():
    with gzip.open(compressed_db, 'rb') as f_in, open(db_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f'Decompressed to: {db_path}')
else:
    print(f'Using existing DB: {db_path}')

conn = sqlite3.connect(str(db_path))
print('SQLite connection opened successfully')

Decompressed to: /home/filipepimentel/Desktop/Thesis code/lrclib-db-dump-20260305T042906Z.sqlite3
SQLite connection opened successfully


In [18]:
tables = pd.read_sql_query(
    'SELECT name FROM sqlite_master WHERE type="table" ORDER BY name',
    conn
)

print('\nTables:')
print(tables)

for t in ['tracks', 'lyrics', 'missing_tracks']:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0]["n"]
    print(f'{t}: {n:,}')


Tables:
                  name
0     _litestream_lock
1      _litestream_seq
2                flags
3               lyrics
4       missing_tracks
5      sqlite_sequence
6               tracks
7           tracks_fts
8    tracks_fts_config
9      tracks_fts_data
10  tracks_fts_docsize
11      tracks_fts_idx
tracks: 23,785,344
lyrics: 24,184,853
missing_tracks: 0


# FMA dataset connection

In [22]:
work_df = small_df[[('artist', 'name'), ('track', 'title'), ('query_key', '')]].copy()
work_df.columns = ['artist', 'title', 'query_key']
work_df = work_df.reset_index().rename(columns={'index': 'track_id'})

unique_queries = work_df[['query_key', 'artist', 'title']].drop_duplicates(subset=['query_key']).copy()
unique_queries['artist_clean'] = unique_queries['artist'].apply(normalize_text)
unique_queries['title_clean'] = unique_queries['title'].apply(normalize_text)

In [23]:
cur = conn.cursor()
cur.execute('DROP TABLE IF EXISTS temp_fma_queries')
cur.execute(
    '''
    CREATE TEMP TABLE temp_fma_queries (
        query_key TEXT PRIMARY KEY,
        artist_clean TEXT,
        title_clean TEXT
    )
    '''
)
cur.executemany(
    'INSERT OR IGNORE INTO temp_fma_queries(query_key, artist_clean, title_clean) VALUES (?, ?, ?)',
    unique_queries[['query_key', 'artist_clean', 'title_clean']].itertuples(index=False, name=None),
)
conn.commit()

In [24]:
matched = pd.read_sql_query(
    '''
    WITH candidates AS (
        SELECT
            q.query_key,
            t.id AS lrclib_track_id,
            t.artist_name AS matched_artist,
            t.name AS matched_title,
            l.source AS lyrics_source,
            l.plain_lyrics,
            l.synced_lyrics,
            CASE
                WHEN l.plain_lyrics IS NOT NULL OR l.synced_lyrics IS NOT NULL THEN 1
                ELSE 0
            END AS lyrics_found
        FROM temp_fma_queries q
        LEFT JOIN tracks t
            ON t.artist_name_lower = q.artist_clean
           AND t.name_lower = q.title_clean
        LEFT JOIN lyrics l
            ON l.id = t.last_lyrics_id
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY query_key
                ORDER BY
                    lyrics_found DESC,
                    CASE WHEN plain_lyrics IS NOT NULL THEN 1 ELSE 0 END DESC,
                    lrclib_track_id
            ) AS rn
        FROM candidates
    )
    SELECT
        query_key,
        lrclib_track_id,
        matched_artist,
        matched_title,
        lyrics_found,
        COALESCE(lyrics_source, 'lrclib') AS lyrics_source,
        COALESCE(plain_lyrics, synced_lyrics) AS lyrics
    FROM ranked
    WHERE rn = 1
    ''',
    conn,
)

results_df = work_df.merge(matched, on='query_key', how='left')
results_df.to_csv('fma_small_lyrics_status_localdb.csv', index=False)

print('\nPreview:')
print(results_df[['track_id', 'artist', 'title', 'lyrics_found', 'lyrics_source']].head())
print('Rows:', len(results_df))
print('Matches found:', int(results_df['lyrics_found'].fillna(0).sum()))
print('Saved: fma_small_lyrics_status_localdb.csv')


Preview:
   track_id                           artist               title  \
0         2                             AWOL                Food   
1         5                             AWOL          This World   
2        10                        Kurt Vile             Freeway   
3       140  Alec K. Redfearn & the Eyesores  Queen Of The Wires   
4       141  Alec K. Redfearn & the Eyesores                Ohio   

   lyrics_found lyrics_source  
0             0        lrclib  
1             0        lrclib  
2             1        lrclib  
3             0        lrclib  
4             0        lrclib  
Rows: 8000
Matches found: 346
Saved: fma_small_lyrics_status_localdb.csv


In [28]:
df = pd.read_csv('fma_small_lyrics_status_localdb.csv')
matched_songs_df = df[df['lyrics_found'].fillna(0).astype(int) == 1].copy()
matched_songs_df.to_csv('fma_small_lyrics_matched_songs.csv', index=False)

print('Songs with lyrics:', len(matched_songs_df))
print('Saved: fma_small_lyrics_matched_songs.csv')


Songs with lyrics: 346
Saved: fma_small_lyrics_matched_songs.csv


# Run VA values on the lyrics

In [ ]:
def prepare_lyrics_text(plain_lyrics, synced_lyrics=None):
    text = plain_lyrics if pd.notna(plain_lyrics) and str(plain_lyrics).strip() else synced_lyrics
    
    if pd.isna(text) or not str(text).strip():
        return None
    
    text = str(text)

    text = re.sub(r"\[\d{1,2}:\d{2}(?:\.\d{1,2})?\]", "", text)
    text = re.sub(r"\[(verse|chorus|bridge|intro|outro|hook)[^\]]*\]", "", text, flags=re.IGNORECASE)
    text = text.replace("\r", "\n")
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text).strip()

    return text if text else None

In [ ]:
matches = pd.read_csv('fma_small_lyrics_matched_songs.csv')

matches['lyrics_text'] = matches.apply(
    lambda row: prepare_lyrics_text(
        row.get('plain_lyrics'), row.get('synced_lyrics')
    ),
    axis=1
)

lyrics_df = matches[matches['lyrics_text'].notna()].copy()
print('Songs with usable lyrics:', len(lyrics_df))

In [ ]:
def split_into_chunks(text, max_words=180, overlap=30):
    words = text.split()
    if not words:
        return []
    
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break
        start = end - overlap
    return chunks

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "RobroKools/vad-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

